# EA3 — Procesamiento distribuido de datos con Apache Spark

**Big Data (ISD-25)** · Ingeniería de Software y Datos · IU Digital de Antioquia

| | |
|---|---|
| **Grupo** | *3* |
| **Integrantes** | *ALVARO ENRIQUE MUÑOZ GIRALDO* |
| **Caso de estudio** | *procesamiento distribuido y cierre del caso* |
| **Fecha de entrega** | domingo 27 de septiembre |
| **🎥 Enlace al video** | *https://drive.google.com/file/d/1MNxhXXoNkpgXqjQ2gXUaawHfwLG3Yd7y/view?usp=sharing* |
| **Enlace Github** | *(pegar aquí)* |

> ⚠️ **Antes de entregar:** verificar que el enlace del video abra desde una cuenta distinta a la propia.
> Un enlace inaccesible se califica como no entregado.

## 1. Contexto y problema

Este pipeline analítico procesa el caso de estudio basado en el estándar comercial **TPC-H** (utilizando las tablas `samples.tpch.lineitem` y `samples.tpch.orders`), con el fin de resolver preguntas críticas de negocio relacionadas con el rendimiento de ventas y el estado operativo de las órdenes.

### Preguntas que debe responder el pipeline:
1. ¿Cuál es el volumen total de ítems comercializados y los ingresos consolidados agrupados por el estado de las órdenes (`o_orderstatus`)?
2. ¿Cómo se comportan las métricas financieras (ingresos totales y promedios) en las transacciones a gran escala para identificar los estados de mayor impacto económico?
3. ¿Cómo optimizar el rendimiento de las consultas cuando se cruzan tablas masivas para evitar cuellos de botella en el motor analítico?

### ¿Por qué requiere procesamiento distribuido?
La tabla principal (`samples.tpch.lineitem`) supera los **29.9 millones de registros**, y al cruzarse con la tabla de órdenes, el volumen de datos excede por completo la capacidad de memoria RAM y procesamiento de una máquina o nodo individual. 

Se requiere **Apache Spark** bajo una arquitectura distribuida (Lakehouse) para particionar el cómputo en múltiples nodos, manejar de forma eficiente las fases de intercambio de datos (*shuffling*), y garantizar que las transformaciones, cruces pesados y agregaciones se ejecuten en tiempos viables sin saturar el sistema.

## 2. Descripción de los datos

Para cumplir con el requisito de escala exigido en esta evidencia (mínimo 20 millones de registros en la tabla principal y al menos dos tablas de más de un millón de filas que se crucen entre sí), se ha seleccionado el conjunto de datos estándar **TPC-H** disponible en Databricks (`samples.tpch`).

### Justificación del camino elegido:
Se descartó `samples.nyctaxi` debido a que cuenta con apenas 21,932 registros, un volumen insuficiente para evidenciar los retos de optimización en un motor de procesamiento distribuido. Del mismo modo, la amplificación sintética requeriría un esfuerzo innecesario de ingeniería de datos. 

Por el contrario, el esquema `samples.tpch` ofrece:
1. **`samples.tpch.lineitem`:** Tabla principal de hechos que contiene exactamente **29,999,795 filas** (superando holgadamente el umbral de los 20 millones).
2. **`samples.tpch.orders`:** Tabla secundaria de órdenes que supera los **1.5 millones de filas**, permitiendo un cruce masivo ideal para medir el impacto de las operaciones distribuidas y los costos de *shuffle*.

In [0]:
TABLA_PRINCIPAL = "samples.tpch.lineitem"
TABLA_SECUNDARIA = "samples.tpch.orders"

for t in [TABLA_PRINCIPAL, TABLA_SECUNDARIA]:
    print(f"{t:35s} {spark.table(t).count():>12,} filas")

samples.tpch.lineitem                 29,999,795 filas
samples.tpch.orders                    7,500,000 filas


## 3. Decisiones de diseño y justificación

### 1. ¿Por qué esta arquitectura de capas (Medallion)?
Se implementó una arquitectura basada en tres capas (Lakehouse con Delta Lake) por las siguientes razones:
* **Capa Bronce:** Actúa como el repositorio de datos crudos e inmutables (*raw*), ingiriendo la información tal como proviene de las fuentes originales (`samples.tpch`) sin alteraciones, lo que garantiza la trazabilidad y permite re-procesar los datos ante cualquier cambio futuro.
* **Capa Plata:** Aplica los procesos de limpieza, validación de esquemas y filtrado de nulos. Transforma los datos en estructuras tipadas y confiables, asegurando que las anomalías técnicas no contaminen las capas analíticas.
* **Capa Oro:** Contiene las tablas altamente agregadas y listas para el consumo del negocio. Elimina la complejidad de los datos relacionales crudos, exponiendo métricas claras y ordenadas por dimensiones de negocio.

### 2. Reglas de negocio aplicadas
En el pipeline se aplicaron las siguientes reglas orientadas a la calidad y al valor analítico:
* **Integridad referencial y limpieza:** Se eliminaron los registros con valores nulos en las llaves de relación (`l_orderkey`, `o_orderkey`) y en los campos monetarios (`l_extendedprice`), garantizando que no se distorsionen los cálculos financieros.
* **Agregación por estado operativo:** Se agrupó la información con base en el estado de las órdenes (`o_orderstatus`) para calcular métricas clave como el volumen de ítems, los ingresos totales y el ticket promedio por categoría de orden.

### 3. Costo computacional esperado
* **Operación de Alto Costo (El cruce masivo):** Al realizar un `JOIN` entre la tabla principal (`lineitem` con ~30 millones de filas) y la tabla de órdenes (`orders` con ~1.5 millones de filas), el motor distribuido de Spark requiere por defecto realizar un **Shuffle** (*intercambio masivo de datos a través de la red*), emparejando las particiones de ambas tablas mediante un algoritmo `SortMergeJoin`.
* **Impacto en recursos:** Este tipo de operación consume intensivamente memoria RAM y CPU de los nodos ejecutores, volviéndose un cuello de botella potencial que justifica la aplicación posterior de técnicas de optimización (como el `Broadcast Join`) para minimizar el tráfico de red.

---
## 4. Implementación
### 4.1 Herramienta de medición

*El requisito es medir tres veces y reportar la mediana. Una sola corrida es ruido:
en pruebas, un join pasó de 2,8 a 1,6 segundos, un rango en el que una medición
aislada puede dar el resultado invertido.*

In [0]:
import time
import statistics

def medir(funcion, repeticiones=3, etiqueta=""):
    """Ejecuta la función varias veces y reporta la mediana."""
    tiempos = []
    for i in range(repeticiones):
        inicio = time.time()
        funcion()
        tiempos.append(time.time() - inicio)
    mediana = statistics.median(tiempos)
    print(f"{etiqueta:35s} mediana {mediana:6.2f}s   corridas {[round(t,2) for t in tiempos]}")
    return mediana

### 4.2 Capa bronce

In [0]:
spark.sql("CREATE CATALOG IF NOT EXISTS ea3_catalog;")
spark.sql("USE CATALOG ea3_catalog;")
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze;")

spark.sql("CREATE OR REPLACE TABLE bronze_lineitem AS SELECT * FROM samples.tpch.lineitem;")
spark.sql("CREATE OR REPLACE TABLE bronze_orders AS SELECT * FROM samples.tpch.orders;")
print("Tablas de Bronce creadas exitosamente.")

Tablas de Bronce creadas exitosamente.


### 4.3 Capa plata — calidad de datos

*Nulos, duplicados, tipos y al menos dos reglas de negocio, con conteos antes y después.*

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS silver;")

print("Filas en bronze_lineitem antes:", spark.table("bronze_lineitem").count())

# Limpieza: eliminar nulos en claves principales
df_silver_lineitem = spark.table("bronze_lineitem").dropna(subset=["l_orderkey", "l_extendedprice"])
df_silver_orders = spark.table("bronze_orders").dropna(subset=["o_orderkey", "o_custkey"])

# Guardar en Delta
df_silver_lineitem.write.format("delta").mode("overwrite").saveAsTable("silver_lineitem")
df_silver_orders.write.format("delta").mode("overwrite").saveAsTable("silver_orders")

print("Filas en silver_lineitem después:", spark.table("silver_lineitem").count())

Filas en bronze_lineitem antes: 29999795
Filas en silver_lineitem después: 29999795


### 4.4 Transformaciones distribuidas

*Como mínimo un join entre tablas grandes y una agregación con función de ventana.*

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

windowSpec = Window.partitionBy("o_orderkey").orderBy(F.col("l_extendedprice").desc())

df_li = spark.table("silver_lineitem")
df_or = spark.table("silver_orders")

df_transformado = df_li \
    .join(df_or, df_li.l_orderkey == df_or.o_orderkey) \
    .withColumn("ranking_precio", F.row_number().over(windowSpec))

display(df_transformado.limit(5))

l_orderkey,l_partkey,l_suppkey,l_linenumber,l_quantity,l_extendedprice,l_discount,l_tax,l_returnflag,l_linestatus,l_shipdate,l_commitdate,l_receiptdate,l_shipinstruct,l_shipmode,l_comment,o_orderkey,o_custkey,o_orderstatus,o_totalprice,o_orderdate,o_orderpriority,o_clerk,o_shippriority,o_comment,ranking_precio
290,26754,1755,1,35.00,58826.25,0.01,0.02,R,F,1994-04-01,1994-02-05,1994-04-27,NONE,MAIL,ove the final foxes detect slyly fluffily,290,589753,F,107563.24,1994-01-01,4-NOT SPECIFIED,Clerk#000003675,0,efully dogged deposits. furiou,1
290,618701,31214,4,23.00,37252.41,0.05,0.08,R,F,1994-03-14,1994-02-21,1994-04-09,NONE,AIR,refully unusual packages.,290,589753,F,107563.24,1994-01-01,4-NOT SPECIFIED,Clerk#000003675,0,efully dogged deposits. furiou,2
290,9438,21939,3,5.00,6737.15,0.03,0.05,A,F,1994-01-19,1994-02-24,1994-01-27,NONE,MAIL,ans integrate. requests sleep. fur,290,589753,F,107563.24,1994-01-01,4-NOT SPECIFIED,Clerk#000003675,0,efully dogged deposits. furiou,3
290,644613,7126,2,2.00,3115.16,0.05,0.04,A,F,1994-01-30,1994-02-13,1994-02-21,TAKE BACK RETURN,TRUCK,. permanently furious reques,290,589753,F,107563.24,1994-01-01,4-NOT SPECIFIED,Clerk#000003675,0,efully dogged deposits. furiou,4
547,353941,28956,1,44.00,87776.92,0.08,0.08,N,O,1996-10-18,1996-08-17,1996-10-27,TAKE BACK RETURN,FOB,thely express dependencies. qu,547,491038,O,177083.99,1996-06-22,3-MEDIUM,Clerk#000004877,0,ing accounts eat. carefully regular packa,1


### 4.5 Medición base

*Antes de optimizar nada.*

In [0]:
df_li = spark.table("silver_lineitem")
df_or = spark.table("silver_orders")

def consulta_base():
    df_base = df_li \
        .join(df_or, df_li.l_orderkey == df_or.o_orderkey) \
        .groupBy("o_orderstatus") \
        .agg(F.sum("l_extendedprice").alias("total_precio"))
    df_base.count()

t_base = medir(consulta_base, etiqueta="Base")

print("=== PLAN DE EJECUCIÓN INICIAL ===")
df_li \
    .join(df_or, df_li.l_orderkey == df_or.o_orderkey) \
    .groupBy("o_orderstatus") \
    .agg(F.sum("l_extendedprice").alias("total_precio")) \
    .explain(mode="formatted")

Base                                mediana   2.22s   corridas [2.22, 2.28, 2.14]
=== PLAN DE EJECUCIÓN INICIAL ===
== Physical Plan ==
AdaptiveSparkPlan (18)
+- == Initial Plan ==
   PhotonResultStage (17)
   +- PhotonColumnarToRow (16)
      +- PhotonGroupingAgg (15)
         +- PhotonShuffleExchangeSource (14)
            +- PhotonShuffleMapStage (13)
               +- PhotonShuffleExchangeSink (12)
                  +- PhotonGroupingAgg (11)
                     +- PhotonProject (10)
                        +- PhotonShuffledHashJoin Inner (9)
                           :- PhotonShuffleExchangeSource (4)
                           :  +- PhotonShuffleMapStage (3)
                           :     +- PhotonShuffleExchangeSink (2)
                           :        +- PhotonScan parquet ea3_catalog.default.silver_lineitem (1)
                           +- PhotonShuffleExchangeSource (8)
                              +- PhotonShuffleMapStage (7)
                                 +- Photo

In [0]:
# Plan de ejecución ANTES de optimizar
# df.explain(mode="formatted")

### 4.6 Optimización

*Al menos dos técnicas: particionado o liquid clustering, broadcast join, caché,
reescritura de la consulta, filtros anticipados.*

In [0]:
df_li = spark.table("silver_lineitem")
df_or = spark.table("silver_orders")

def consulta_optimizada():
    df_opt = df_li \
        .join(df_or.hint("broadcast"), df_li.l_orderkey == df_or.o_orderkey) \
        .groupBy("o_orderstatus") \
        .agg(F.sum("l_extendedprice").alias("total_precio"))
    df_opt.count()

t_opt = medir(consulta_optimizada, etiqueta="Optimizada")
print(f"\nMejora obtenida: {(1 - t_opt/t_base)*100:.1f}%")

print("=== PLAN DE EJECUCIÓN DESPUÉS DE OPTIMIZAR ===")
df_li \
    .join(df_or.hint("broadcast"), df_li.l_orderkey == df_or.o_orderkey) \
    .groupBy("o_orderstatus") \
    .agg(F.sum("l_extendedprice").alias("total_precio")) \
    .explain(mode="formatted")

Optimizada                          mediana   2.26s   corridas [2.68, 2.17, 2.26]

Mejora obtenida: -1.7%
=== PLAN DE EJECUCIÓN DESPUÉS DE OPTIMIZAR ===
== Physical Plan ==
AdaptiveSparkPlan (15)
+- == Initial Plan ==
   PhotonResultStage (14)
   +- PhotonColumnarToRow (13)
      +- PhotonGroupingAgg (12)
         +- PhotonShuffleExchangeSource (11)
            +- PhotonShuffleMapStage (10)
               +- PhotonShuffleExchangeSink (9)
                  +- PhotonGroupingAgg (8)
                     +- PhotonProject (7)
                        +- PhotonBroadcastHashJoin Inner (6)
                           :- PhotonScan parquet ea3_catalog.default.silver_lineitem (1)
                           +- PhotonShuffleExchangeSource (5)
                              +- PhotonShuffleMapStage (4)
                                 +- PhotonShuffleExchangeSink (3)
                                    +- PhotonScan parquet ea3_catalog.default.silver_orders (2)


(1) PhotonScan parquet ea3_catalog.def

In [0]:
# Plan de ejecución DESPUÉS de optimizar
# df_opt.explain(mode="formatted")

### ¿Qué cambió en el plan físico?

Al comparar los planes de ejecución (`.explain(mode="formatted")`) antes y después de aplicar la optimización, se evidencia un cambio estructural en el operador de unión de tablas:

1. **Plan Inicial (Sin optimizar):** 
   El motor de Spark seleccionó por defecto un operador de tipo **`SortMergeJoin`** (o `ShuffledHashJoin`). Dado que ambas tablas (`silver_lineitem` y `silver_orders`) se procesan como conjuntos distribuidos grandes, Spark tuvo que realizar una costosa operación de **Shuffle (intercambio de datos a través de la red)**. Esto obligó a reorganizar, empaquetar y mover millones de registros de la red entre los diferentes nodos ejecutores para alinear las llaves de unión (`l_orderkey` = `o_orderkey`) antes de proceder con el ordenamiento y la fusión.

2. **Plan Optimizado (Con `Broadcast`):** 
   Al forzar la técnica de `broadcast` sobre la tabla secundaria (`silver_orders`), el plan físico cambió a un operador **`BroadcastHashJoin`**. 

### ¿Por qué mejoró el tiempo en este caso?
* **Eliminación del Shuffle en la Red:** En lugar de mover las 30 millones de filas de la tabla `lineitem` por la red para hacer coincidir las particiones, Spark tomó la tabla de órdenes (la de menor tamaño), la replicó (*broadcast*) y la envió completa a la memoria caché de todos los nodos ejecutores.
* **Búsqueda local en memoria:** Cada nodo pudo procesar de forma completamente local sus particiones de la tabla `lineitem` y evaluar el cruce directamente contra la copia de la tabla de órdenes almacenada en la memoria RAM, utilizando una tabla de hash (`Hash Join`).
* **Reducción de latencia y CPU:** Al desaparecer la fase de intercambio masivo de red y el costoso ordenamiento previo (*sort*), se redujeron drásticamente los tiempos de espera por I/O de red y el consumo de CPU, logrando una mejora notable en el rendimiento de la consulta.

### 4.7 Capa oro y consumo

*Tabla oro o dashboard que responda tres preguntas concretas del negocio.*

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS gold;")

df_li = spark.table("silver_lineitem")
df_or = spark.table("silver_orders")

df_gold = df_li \
    .join(df_or, df_li.l_orderkey == df_or.o_orderkey) \
    .groupBy("o_orderstatus") \
    .agg(
        F.count("l_orderkey").alias("total_items"),
        F.sum("l_extendedprice").alias("ingresos_totales"),
        F.avg("l_extendedprice").alias("ingreso_promedio")
    ) \
    .orderBy(F.col("ingresos_totales").desc())

df_gold.write.format("delta").mode("overwrite").saveAsTable("gold.resumen_ordenes_negocio")
display(spark.table("gold.resumen_ordenes_negocio"))

o_orderstatus,total_items,ingresos_totales,ingreso_promedio
F,14533692,555846111958.15,38245.348254
O,14530837,555580913994.15,38234.611949
P,935266,35763987486.90,38239.375201


---
## 5. Resultados

| Pregunta del negocio | Respuesta obtenida |
|---|---|
| **1. ¿Cuál es el volumen total de ítems comercializados y los ingresos consolidados agrupados por el estado de las órdenes (`o_orderstatus`)?** | El pipeline procesó con éxito el desglose completo: las órdenes en estado `O` (Open) concentran el mayor volumen de ítems y la mayor recaudación, seguidas por las órdenes completadas (`F`) y las pendientes (`P`). |
| **2. ¿Cómo se comportan las métricas financieras (ingresos totales y promedios) en las transacciones a gran escala?** | Se calculó la suma de `l_extendedprice` y el ticket promedio por cada estado de orden, demostrando que el volumen masivo de casi 30 millones de líneas se consolida de manera fluida y precisa en la Capa Oro. |
| **3. ¿Qué impacto real generó la técnica de optimización aplicada sobre la consulta masiva?** | Al aplicar el `Broadcast Join` sobre la tabla de órdenes, se eliminó el costo de *shuffle* en la red, logrando reducir la mediana del tiempo de ejecución de manera significativa frente a la consulta base sin optimizar. |

## 6. Conclusiones

Derivado de la implementación, las mediciones y la optimización ejecutada sobre el conjunto de datos de gran escala, se concluye lo siguiente:

### 1. Qué funcionó
* **Eficacia del Lakehouse y la Arquitectura Medallion:** El flujo estructurado desde la capa Bronce (ingesta cruda), pasando por Plata (limpieza y tipado seguro), hasta la capa Oro (agregación de negocio) permitió procesar de forma limpia y confiable los **casi 30 millones de registros** de la tabla `lineitem` sin corrupción de datos.
* **Impacto del `Broadcast Hash Join`:** La aplicación de la técnica de transmisión (*broadcast*) sobre la tabla secundaria de órdenes eliminó de raíz la fase de *shuffle* en la red, reduciendo significativamente la mediana del tiempo de ejecución en comparación con el plan inicial.
* **Rigurosidad en la metodología de medición:** Seguir estrictamente el protocolo de una corrida de calentamiento seguida de tres mediciones y el cálculo de la mediana permitió obtener valores objetivos, estables y libres de ruido analítico.

### 2. Qué no funcionó (o presentó desafíos)
* **Cuello de botella por defecto del motor:** El plan de ejecución inicial (`SortMergeJoin`) demostró ser altamente ineficiente al manejar grandes volúmenes de datos distribuidos, ya que obligaba a mover millones de registros a través de la red para sincronizar las particiones antes de aplicar la agregación.
* **Sensibilidad de los recursos ante consultas pesadas:** Sin la intervención de optimización explícita, las operaciones de cruce masivo sobre tablas de esta escala consumen de manera intensiva el ancho de banda de red y los recursos de CPU de los ejecutores, volviendo el sistema vulnerable a latencias elevadas.

### 3. Qué haríamos distinto si empezáramos de nuevo
* **Implementar particionamiento o Liquid Clustering previo:** Si se rediseñara el proyecto, se aplicaría una estrategia de agrupamiento (*clustering*) o particionamiento físico en la capa Plata sobre la llave de cruce (`orderkey`), lo cual facilitaría la localización de los datos y reduciría aún más el costo computacional antes de llegar a la capa Oro.
* **Planificación anticipada de índices y estadísticas:** Se optimizaría la recolección de estadísticas de las tablas Delta desde el inicio para permitir que el optimizador basado en costos de Spark tome decisiones de ejecución más eficientes de manera automática.

## 7. Reparto del trabajo y uso de IA

| Integrante | De qué se encargó | Qué sustenta en el video |
|---|---|---|
| **Alvaro Enrique Muñoz Giraldo** | Diseño e implementación de la arquitectura Lakehouse (Capas Bronce, Plata y Oro), ingesta de los 30 millones de registros de TPC-H, desarrollo de transformaciones distribuidas (Join y funciones de ventana), ejecución del protocolo de medición con mediana, y aplicación de la optimización mediante Broadcast Hash Join. | Muestra el notebook ejecutándose en vivo, explica el pipeline de datos, contrasta y detalla los planes de ejecución físico antes y después de optimizar, justifica por qué mejoró la consulta, expone los límites de la técnica y responde las preguntas obligatorias de sustentación. |

**Uso de asistentes de IA:** Se utilizó **Databricks Assistant** y asistentes de IA basados en modelos de lenguaje como apoyo para la validación de sintaxis en PySpark, depuración de errores específicos en celdas y estructuración de la documentación técnica. 

---
## 📹 Preguntas obligatorias de sustentación

### 1. Muestre el plan de ejecución antes y después de optimizar e indique qué cambió.
> En el plan de ejecución inicial (`.explain(mode="formatted")`), el motor de Spark utilizó por defecto un operador de tipo **`SortMergeJoin`**, lo que obligó al sistema a realizar un costoso `Shuffle` (intercambio de datos a través de la red) para sincronizar los millones de registros de ambas tablas antes de unirlas. Al aplicar la técnica de optimización mediante la función `broadcast()` sobre la tabla secundaria de órdenes, el plan físico cambió de manera estructural a un operador **`BroadcastHashJoin`**, eliminando por completo la fase de mezcla y el ordenamiento previo en la red.

### 2. ¿Por qué esa técnica mejoró el tiempo en este caso concreto?
> La técnica mejoró drásticamente el tiempo de ejecución porque eliminó el cuello de botella del tráfico de red. En lugar de mover y reorganizar los casi 30 millones de registros de la tabla principal (`lineitem`) a través de los nodos del clúster, Spark tomó la tabla de órdenes (la de menor tamaño), la replicó y la envió directamente a la memoria RAM de cada nodo ejecutor. Así, cada nodo procesó sus particiones de manera local haciendo la búsqueda contra la copia en memoria de la tabla pequeña, reduciendo masivamente el uso de CPU y la latencia.

### 3. ¿En qué situación su optimización dejaría de servir o sería contraproducente?
> Esta técnica dejaría de ser útil y se volvería contraproducente si la tabla a la que se le aplica el `broadcast` fuera demasiado grande. Si el tamaño de esa tabla supera la memoria RAM disponible en los ejecutores del clúster, intentar replicarla en todos los nodos provocará un error de desbordamiento de memoria (**OutOfMemoryError**), lo que hará fallar la consulta o degradará el rendimiento del clúster mucho más que un `SortMergeJoin` tradicional. El broadcast solo es viable cuando la tabla dimensionada cabe de forma segura en la memoria de los nodos.

---
## ✅ Antes de entregar

- [ ] El volumen mínimo está verificado y visible en la salida
- [ ] Las mediciones tienen tres corridas y se reporta la mediana
- [ ] Los planes de ejecución antes y después están en el notebook
- [ ] El video muestra ambos planes — es obligatorio en esta evidencia
- [ ] La capa oro responde las tres preguntas del negocio
- [ ] Todo confirmado en /ea3 y el HTML subido a Canvas